# ETL Base Template — DuckDB Workflow

**Template ID:** ETL-BASE  
**Workflow layer:** `source → raw → staging → curated → output`  
**Primary mode:** Notebook + SQL via DuckDB Python API

## Purpose

Notebook-first ETL template for DuckDB pipelines. Ingest a source file, land it in **raw**, clean and standardize into **staging**, build a business-ready **curated** table, validate each layer, and export to **output**.

Follows the layered workflow convention:

```text
source → raw → staging → curated → output
```

## How to use

1. Copy this notebook into your project (or run it from this repo).
2. Update the **Project configuration** cell with paths, table names, and column lists.
3. Run all cells top to bottom.
4. Review validation results and the **Reconciliation summary** before publishing.
5. Record **Assumptions** and **Next actions** at the end.

**Example tables in this template:** `raw.raw_orders`, `staging.stg_orders`, `curated.fct_orders`  
**Target users:** analysts, data engineers, GIS users, SQL users, Python users.

## 2. Project configuration

Edit the variables below for your dataset. The default example ingests DuckDB's public TPC-H `lineitem` Parquet and maps it to an orders-shaped model.

In [ ]:
from pathlib import Path

# --- paths ---
PROJECT_ROOT = Path("..").resolve()  # repo root when notebook lives in notebooks/
DB_PATH = PROJECT_ROOT / "work.duckdb"
RAW_INPUT_PATH = "https://shell.duckdb.org/data/tpch/0_01/parquet/lineitem.parquet"  # source file or URL
OUTPUT_PATH = PROJECT_ROOT / "data" / "output" / "fct_orders.parquet"

# --- tables (schema.table) ---
RAW_TABLE_NAME = "raw.raw_orders"
STAGING_TABLE_NAME = "staging.stg_orders"
CURATED_TABLE_NAME = "curated.fct_orders"

# --- source format (used when registering data in this notebook) ---
# Options: csv | parquet | json | shapefile | geojson | geoparquet | gdb
SOURCE_FORMAT = "parquet"

# --- validation ---
KEY_COLUMNS = ["order_id"]  # business / natural keys for uniqueness checks
REQUIRED_COLUMNS = ["order_id", "order_date", "amount"]  # must not be NULL in staging

# --- ingest controls ---
REGISTER_SOURCE = True  # set False when raw table already exists
RAW_ROW_LIMIT = 50_000  # limit rows for notebook demos; set None for full ingest
PREVIEW_LIMIT = 10

## 3. Imports

In [ ]:
import duckdb
from IPython.display import display

## 4. Connect to DuckDB

Opens (or creates) a persistent database and ensures workflow schemas exist.

In [ ]:
con = duckdb.connect(str(DB_PATH))

for schema in ("raw", "staging", "curated"):
    con.execute(f'CREATE SCHEMA IF NOT EXISTS "{schema}";')

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"Connected to: {DB_PATH}")
display(con.sql("SELECT version() AS duckdb_version").df())

## 5. Load extensions

Load extensions required for your source format. Remote URLs need `httpfs`; spatial formats need `spatial`.

In [ ]:
EXTENSIONS = ["httpfs", "json"]

if SOURCE_FORMAT in {"shapefile", "geojson", "geoparquet", "gdb"}:
    EXTENSIONS.append("spatial")

if str(RAW_INPUT_PATH).startswith(("http://", "https://")):
    if "httpfs" not in EXTENSIONS:
        EXTENSIONS.insert(0, "httpfs")

for ext in dict.fromkeys(EXTENSIONS):  # preserve order, remove duplicates
    con.execute(f"INSTALL {ext};")
    con.execute(f"LOAD {ext};")
    print(f"Loaded extension: {ext}")

## 6. Register or ingest source data

Materialize the source into the `raw` layer with minimal transformation. **Skip this cell** (`REGISTER_SOURCE = False`) when the table already exists from a prior ingest.

For spatial sources, use `ST_Read()` — see `docs/03_spatial_ingestion/`.

In [ ]:
source_path = Path(RAW_INPUT_PATH).as_posix() if not str(RAW_INPUT_PATH).startswith(("http://", "https://")) else str(RAW_INPUT_PATH)
limit_clause = f"LIMIT {RAW_ROW_LIMIT}" if RAW_ROW_LIMIT else ""

if REGISTER_SOURCE:
    if SOURCE_FORMAT == "csv":
        ingest_sql = f"""
        CREATE OR REPLACE TABLE {RAW_TABLE_NAME} AS
        SELECT *
        FROM read_csv_auto('{source_path}', header = true, sample_size = -1)
        {limit_clause};
        """
    elif SOURCE_FORMAT == "parquet":
        ingest_sql = f"""
        CREATE OR REPLACE TABLE {RAW_TABLE_NAME} AS
        SELECT *
        FROM read_parquet('{source_path}')
        {limit_clause};
        """
    elif SOURCE_FORMAT == "json":
        ingest_sql = f"""
        CREATE OR REPLACE TABLE {RAW_TABLE_NAME} AS
        SELECT *
        FROM read_json_auto('{source_path}')
        {limit_clause};
        """
    elif SOURCE_FORMAT in {"shapefile", "geojson", "geoparquet", "gdb"}:
        ingest_sql = f"""
        CREATE OR REPLACE TABLE {RAW_TABLE_NAME} AS
        SELECT *
        FROM ST_Read('{source_path}')
        {limit_clause};
        """
    else:
        raise ValueError(f"Unsupported SOURCE_FORMAT: {SOURCE_FORMAT}")

    con.execute(ingest_sql)
    raw_count = con.sql(f"SELECT COUNT(*) AS row_count FROM {RAW_TABLE_NAME}").df().row_count.iloc[0]
    print(f"Registered source → {RAW_TABLE_NAME} ({raw_count:,} rows)")
else:
    print(f"Skipping registration. Using existing table: {RAW_TABLE_NAME}")

## 7. Inspect raw data

Preview rows, schema, and row count before building staging.

In [ ]:
raw_preview_sql = f"""
SELECT *
FROM {RAW_TABLE_NAME}
LIMIT {PREVIEW_LIMIT};
"""

raw_schema_sql = f"DESCRIBE {RAW_TABLE_NAME};"

raw_count_sql = f"""
SELECT COUNT(*) AS row_count
FROM {RAW_TABLE_NAME};
"""

print("--- Preview ---")
display(con.sql(raw_preview_sql).df())

print("--- Schema ---")
display(con.sql(raw_schema_sql).df())

print("--- Row count ---")
display(con.sql(raw_count_sql).df())

## 8. Create staging table

Clean, rename, cast, and standardize columns from `raw` into `staging`. The example maps TPC-H `lineitem` fields to an orders-shaped staging model.

Adapt the `SELECT` list when your source schema differs.

In [ ]:
create_staging_sql = f"""
CREATE OR REPLACE TABLE {STAGING_TABLE_NAME} AS
SELECT
  l_orderkey::BIGINT AS order_id,
  l_linenumber::INTEGER AS line_number,
  l_suppkey::BIGINT AS supplier_id,
  CAST(l_shipdate AS DATE) AS order_date,
  TRY_CAST(l_extendedprice AS DOUBLE) AS amount,
  TRY_CAST(l_quantity AS DOUBLE) AS quantity,
  CASE
    WHEN l_returnflag = 'R' THEN 'returned'
    WHEN l_linestatus = 'F' THEN 'fulfilled'
    ELSE 'open'
  END AS order_status,
  l_shipmode AS ship_mode
FROM {RAW_TABLE_NAME}
WHERE l_orderkey IS NOT NULL;
"""

con.execute(create_staging_sql)

stg_count = con.sql(f"SELECT COUNT(*) AS row_count FROM {STAGING_TABLE_NAME}").df().row_count.iloc[0]
print(f"Created {STAGING_TABLE_NAME} ({stg_count:,} rows)")

display(con.sql(f"SELECT * FROM {STAGING_TABLE_NAME} LIMIT {PREVIEW_LIMIT}").df())

## 9. Validate staging table

Run row-count, required-field null, and primary-key uniqueness checks. Zero failing rows = pass.

In [ ]:
stg_row_count_sql = f"""
SELECT COUNT(*) AS row_count
FROM {STAGING_TABLE_NAME};
"""

required_null_predicates = " OR ".join(
    f'"{col}" IS NULL' for col in REQUIRED_COLUMNS
)
required_null_sql = f"""
SELECT
  COUNT(*) AS rows_with_required_nulls
FROM {STAGING_TABLE_NAME}
WHERE {required_null_predicates};
"""

key_select = ", ".join(f'"{col}"' for col in KEY_COLUMNS)
duplicate_key_sql = f"""
SELECT
  {key_select},
  COUNT(*) AS duplicate_count
FROM {STAGING_TABLE_NAME}
GROUP BY {key_select}
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC
LIMIT 20;
"""

print("--- Staging row count ---")
display(con.sql(stg_row_count_sql).df())

print("--- Required-field null check (0 = pass) ---")
display(con.sql(required_null_sql).df())

print(f"--- Duplicate key check on {KEY_COLUMNS} ---")
dupes = con.sql(duplicate_key_sql).df()
if dupes.empty:
    print(f"PASS: no duplicate key combinations for {KEY_COLUMNS}")
else:
    display(dupes)

## 10. Create curated table

Apply business rules and build an analysis-ready fact table. The example keeps non-returned line items with positive amounts.

In [ ]:
create_curated_sql = f"""
CREATE OR REPLACE TABLE {CURATED_TABLE_NAME} AS
SELECT
  order_id,
  line_number,
  supplier_id,
  order_date,
  amount,
  quantity,
  order_status,
  ship_mode,
  DATE_TRUNC('month', order_date) AS order_month
FROM {STAGING_TABLE_NAME}
WHERE order_status <> 'returned'
  AND amount IS NOT NULL
  AND amount > 0;
"""

con.execute(create_curated_sql)

fct_count = con.sql(f"SELECT COUNT(*) AS row_count FROM {CURATED_TABLE_NAME}").df().row_count.iloc[0]
print(f"Created {CURATED_TABLE_NAME} ({fct_count:,} rows)")

display(con.sql(f"SELECT * FROM {CURATED_TABLE_NAME} LIMIT {PREVIEW_LIMIT}").df())

## 11. Validate curated table

Confirm curated row volume, required fields, and measure sanity before export.

In [ ]:
curated_row_count_sql = f"""
SELECT COUNT(*) AS row_count
FROM {CURATED_TABLE_NAME};
"""

curated_required_null_sql = f"""
SELECT
  COUNT(*) AS rows_with_required_nulls
FROM {CURATED_TABLE_NAME}
WHERE {required_null_predicates};
"""

curated_measure_sql = f"""
SELECT
  COUNT(*) AS row_count,
  ROUND(SUM(amount), 2) AS total_amount,
  ROUND(AVG(amount), 4) AS avg_amount,
  MIN(order_date) AS min_order_date,
  MAX(order_date) AS max_order_date
FROM {CURATED_TABLE_NAME};
"""

curated_status_sql = f"""
SELECT
  order_status,
  COUNT(*) AS row_count,
  ROUND(SUM(amount), 2) AS total_amount
FROM {CURATED_TABLE_NAME}
GROUP BY order_status
ORDER BY row_count DESC;
"""

print("--- Curated row count ---")
display(con.sql(curated_row_count_sql).df())

print("--- Required-field null check (0 = pass) ---")
display(con.sql(curated_required_null_sql).df())

print("--- Measure summary ---")
display(con.sql(curated_measure_sql).df())

print("--- Status distribution ---")
display(con.sql(curated_status_sql).df())

## 12. Export output

Write the curated table to `output` as Parquet. Change the `COPY` format for CSV or GeoParquet when needed — see `docs/10_export/`.

In [ ]:
output_path = OUTPUT_PATH.resolve().as_posix()

export_sql = f"""
COPY (
  SELECT
    order_id,
    line_number,
    supplier_id,
    order_date,
    order_month,
    amount,
    quantity,
    order_status,
    ship_mode
  FROM {CURATED_TABLE_NAME}
) TO '{output_path}'
(FORMAT PARQUET, COMPRESSION ZSTD);
"""

con.execute(export_sql)

export_verify_sql = f"""
SELECT COUNT(*) AS exported_row_count
FROM read_parquet('{output_path}');
"""

print(f"Exported → {OUTPUT_PATH}")
display(con.sql(export_verify_sql).df())

## 13. Reconciliation summary

Compare row counts and key aggregates across layers. Investigate any unexpected deltas before publishing.

In [ ]:
layer_counts_sql = f"""
SELECT '{RAW_TABLE_NAME}' AS layer, COUNT(*) AS row_count
FROM {RAW_TABLE_NAME}
UNION ALL
SELECT '{STAGING_TABLE_NAME}', COUNT(*)
FROM {STAGING_TABLE_NAME}
UNION ALL
SELECT '{CURATED_TABLE_NAME}', COUNT(*)
FROM {CURATED_TABLE_NAME}
UNION ALL
SELECT 'output file', COUNT(*)
FROM read_parquet('{output_path}');
"""

aggregate_reconciliation_sql = f"""
WITH source_agg AS (
  SELECT
    SUM(amount) AS total_amount,
    COUNT(*) AS row_count
  FROM {STAGING_TABLE_NAME}
  WHERE order_status <> 'returned'
    AND amount IS NOT NULL
    AND amount > 0
),
target_agg AS (
  SELECT
    SUM(amount) AS total_amount,
    COUNT(*) AS row_count
  FROM {CURATED_TABLE_NAME}
)
SELECT 'staging (eligible)' AS layer, s.total_amount, s.row_count
FROM source_agg AS s
UNION ALL
SELECT 'curated', t.total_amount, t.row_count
FROM target_agg AS t
UNION ALL
SELECT 'delta', t.total_amount - s.total_amount, t.row_count - s.row_count
FROM source_agg AS s, target_agg AS t;
"""

print("--- Layer row counts ---")
display(con.sql(layer_counts_sql).df())

print("--- Amount reconciliation (staging eligible vs curated) ---")
display(con.sql(aggregate_reconciliation_sql).df())

## 14. Assumptions

Document decisions made in this run. Replace the bullets below with your project-specific notes.

- **Source:** DuckDB public TPC-H `lineitem` Parquet at `shell.duckdb.org` (`RAW_INPUT_PATH`) stands in for an orders feed; swap for your CSV, API extract, or spatial source.
- **Grain:** One row per order line (`order_id` + `line_number`); `KEY_COLUMNS` uses `order_id` only — add `line_number` for line-level uniqueness.
- **Column mapping:** `l_suppkey` → `supplier_id`; `l_extendedprice` → `amount`; `l_shipdate` → `order_date`.
- **Business rules:** Curated layer excludes `returned` lines and non-positive amounts.
- **Volume:** `RAW_ROW_LIMIT` caps ingest for notebook demos; remove for production loads.
- **Spatial:** Not used in this example; set `SOURCE_FORMAT` to `shapefile`, `geojson`, `geoparquet`, or `gdb` and load `spatial` for GIS workflows.

## 15. Next actions

Map this ETL run to the next workflow step. Check the boxes that apply to your project.

- [ ] **EDA** — profile raw or staging tables (`notebooks/00_eda_base.ipynb`)
- [ ] **Cleaning** — deduplication, casting, text normalization (`docs/06_cleaning/`)
- [ ] **Dimensions** — build `curated.dim_*` and join in fact tables (`docs/07_transformation/build_fact_table.md`)
- [ ] **Validation** — referential integrity, domain checks, validation summary (`docs/09_validation/`)
- [ ] **Spatial** — ingest Shapefile, GeoParquet, GeoJSON, or File Geodatabase (`docs/03_spatial_ingestion/`)
- [ ] **Export variants** — partitioned Parquet, CSV, GeoParquet (`docs/10_export/`)
- [ ] **Productionize** — move SQL to `sql/` scripts and helpers in `python/`

---

_Close the DuckDB connection when finished, or leave it open for the next notebook in the session._

In [ ]:
# con.close()
# print("Connection closed.")